In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml import ExperimentTracker
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


# Columns that are NOT behavioural features. ``churned`` is an OUTCOME label:
# feeding it into the clustering would make the "behavioural" segments
# partly a split on the answer, and a standardised binary column dominates
# Euclidean distance. It stays in the DataFrame for post-hoc profiling only
# (e.g. "what is the churn rate of each segment?").
NON_FEATURE_COLS: tuple[str, ...] = ("customer_id", "churned")


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is 50,000 Singapore e-commerce customers. The
    7 numeric behavioural features are revenue, order count, average order
    value, recency, tenure, satisfaction (a 1-5 survey score) and number of
    returns (a 0-6 count). The last two are discrete — after standardising
    they form a few stacked "bands", which is worth remembering when you
    read cluster shapes. ``churned`` is returned in the DataFrame but is
    deliberately excluded from the features (see NON_FEATURE_COLS).
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in NON_FEATURE_COLS
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every clustering technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_1 lesson logs its sweep + final-fit metrics to a single
# SQLite store so students can compare runs across techniques after the
# lesson group ends. Mirrors the M5 ex_1 pattern (autoencoder zoo).

CLUSTERING_DB = "sqlite:///mlfp04_ex1_clustering.db"
EXPERIMENT_NAME = "m4_clustering_zoo"


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the clustering ExperimentTracker (kailash-ml)."""
    tracker = await ExperimentTracker.create(store_url=CLUSTERING_DB)
    return tracker, EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block. See ``rules/patterns.md`` § "Async Resource Cleanup".
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series.

    The tracker rejects NaN/inf metric values, and an undefined metric (e.g.
    the silhouette of a 1-cluster DBSCAN result) is a legitimate outcome, so
    non-finite scalars are skipped with an explicit message, not hidden.
    """
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            if not np.isfinite(float(value)):
                print(f"  [tracker] not logging {name}={value} (undefined metric)")
                continue
            await run.log_metric(name, float(value))
        if series_metrics:
            for name, values in series_metrics.items():
                for step, value in enumerate(values, start=1):
                    await run.log_metric(name, float(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )


# ── shared/mlfp04/ex_2.py ──
"""
Shared infrastructure for MLFP04 Exercise 2 — EM and Gaussian Mixture Models.

Contains: synthetic GMM data generation, Singapore e-commerce loader, scaler,
scoring helpers, and OUTPUT_DIR management.

Technique-specific code (the EM E-step / M-step from scratch, BIC/AIC sweeps,
covariance-type comparison, mixture-of-experts gating) does NOT live here.
It lives in the per-technique files under modules/mlfp04/solutions/ex_2/.
"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex2_gmm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 2D DATA — 3 well-separated Gaussians
# ════════════════════════════════════════════════════════════════════════
#
# Used to validate the from-scratch EM implementation. Three well-separated
# components make convergence obvious and let students verify the recovered
# parameters against the ground truth.

TRUE_MEANS: np.ndarray = np.array([[0.0, 0.0], [5.0, 2.0], [2.0, 6.0]])
TRUE_COVS: np.ndarray = np.array(
    [
        [[1.0, 0.3], [0.3, 0.8]],
        [[0.8, -0.2], [-0.2, 1.2]],
        [[1.5, 0.0], [0.0, 0.5]],
    ]
)
TRUE_WEIGHTS: np.ndarray = np.array([0.4, 0.35, 0.25])
N_SYNTH: int = 600


def make_synthetic_gmm(
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw N_SYNTH points from the 3-component GMM defined by TRUE_*.

    Returns (X, z_true) where z_true is the ground-truth component index.
    """
    rng = np.random.default_rng(seed)
    n_per = (TRUE_WEIGHTS * N_SYNTH).astype(int)
    n_per[-1] = N_SYNTH - n_per[:-1].sum()

    parts: list[np.ndarray] = []
    labels: list[int] = []
    for k, (mean, cov, n) in enumerate(zip(TRUE_MEANS, TRUE_COVS, n_per)):
        parts.append(rng.multivariate_normal(mean, cov, n))
        labels.extend([k] * n)

    X = np.vstack(parts)
    z = np.array(labels)

    # Shuffle so the order does not leak the label
    idx = rng.permutation(N_SYNTH)
    return X[idx], z[idx]


# ════════════════════════════════════════════════════════════════════════
# REAL DATA — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════
#
# We reuse the MLFP03 e-commerce customer dataset (50,000 Singapore
# customers, 7 numeric behavioural features) for every real-data task in
# this exercise. ``churned`` is an outcome label, so it is kept in the
# DataFrame for post-hoc segment profiling but excluded from the features. Segmentation is the business
# frame: the GMM will propose soft customer segments that marketing can
# score on expected value.


# ``churned`` is the outcome we may later want to explain — never a
# segmentation input.
NON_FEATURE_COLS: tuple[str, ...] = ("customer_id", "churned")


def load_customers_scaled() -> (
    tuple[np.ndarray, pl.DataFrame, list[str], StandardScaler]
):
    """Return (X_scaled, customers_df, feature_cols, scaler).

    The DataFrame and feature_cols are returned so technique files can
    join cluster labels back onto the original rows for segment profiling.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in NON_FEATURE_COLS
    ]
    customers = customers.drop_nulls(subset=feature_cols)
    X, _, _ = to_sklearn_input(customers, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, customers, feature_cols, scaler


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING — for BIC/AIC interpretation
# ════════════════════════════════════════════════════════════════════════


def count_gmm_params(n_components: int, n_features: int, cov_type: str) -> int:
    """Number of free parameters in a GMM given components, features, cov type.

    Used to explain the BIC/AIC ranking across covariance types.
    """
    d = n_features
    k = n_components
    if cov_type == "full":
        return k * (d * (d + 1) // 2 + d + 1) - 1
    if cov_type == "tied":
        return d * (d + 1) // 2 + k * (d + 1) - 1
    if cov_type == "diag":
        return k * (2 * d + 1) - 1
    if cov_type == "spherical":
        return k * (d + 2) - 1
    raise ValueError(f"Unknown cov_type: {cov_type}")


# ════════════════════════════════════════════════════════════════════════
# SCORING HELPERS
# ════════════════════════════════════════════════════════════════════════


def safe_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    """Silhouette with a graceful fallback when only one cluster is present."""
    if len(set(labels.tolist())) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 2.4: Mixture of Experts — Soft vs Hard Assignments
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Read GMM soft assignments as intent vectors, not class labels
#   - Measure assignment confidence with max-probability and entropy
#   - Identify boundary customers that hard clustering would bury
#   - Explain Mixture of Experts as input-dependent gating (g_k(x))
#   - Connect classical MoE to Sparse MoE in modern LLMs (Mixtral)
#
# PREREQUISITES: 03_covariance_types.py (BIC-optimal K on customer data)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — MoE as GMM with input-dependent mixing
#   2. Build — soft_vs_hard analysis + simple MoE gate demo
#   3. Train — fit the BIC-optimal GMM and extract soft responsibilities
#   4. Visualise — confidence histogram + segment profile
#   5. Apply — Singapore C2C marketplace personalised listing ranking
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
from sklearn.mixture import GaussianMixture

from kailash_ml import ModelVisualizer

# Cross-exercise import: tracker helpers live in ex_1.shared so every M4
# unsupervised technique logs to the same `m4_clustering_zoo` experiment.



### Kailash-ML ExperimentTracker — every clustering run logs here


In [ ]:
tracker, exp_name = setup_engines()



## THEORY — From GMM to Mixture of Experts


In [ ]:
#
#   GMM:  P(x)   = Sum_k pi_k * N(x | mu_k, Sigma_k)    (fixed pi_k)
#   MoE:  P(y|x) = Sum_k g_k(x) * f_k(x)                (input-dep g_k)
#
# g_k(x) is a softmax over gating logits:
#     g_k(x) = exp(w_k^T x) / Sum_j exp(w_j^T x)
#
# Mixtral 8x7B uses this idea at LLM scale: 8 feed-forward experts per
# layer, top-2 routing per token by a single linear router. Attention is
# shared, so ~46.7B total params (not 56B) and ~12.9B active per token.
# Same gating equation.



## TASK 2 — BUILD: soft_vs_hard analysis + gate demo


Summarise a responsibility matrix R into a confidence profile.


A 2-expert softmax gate: route by the sign of the first feature.


In [ ]:
def soft_vs_hard(soft_probs: np.ndarray) -> dict[str, float]:
    # TODO: max_probs = soft_probs.max(axis=1)
    max_probs = ____
    # TODO: entropy = -sum(p * log(p+eps)) along axis=1
    entropy = ____
    return {
        "confident": float((max_probs > 0.95).mean()),
        "moderate": float(((max_probs > 0.7) & (max_probs <= 0.95)).mean()),
        "ambiguous": float(((max_probs > 0.5) & (max_probs <= 0.7)).mean()),
        "uncertain": float((max_probs <= 0.5).mean()),
        "mean_entropy": float(entropy.mean()),
        "max_entropy": float(np.log(soft_probs.shape[1])),
    }


def simple_moe_gate(X: np.ndarray, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    logits = np.column_stack([X[:, 0], -X[:, 0]]) + 0.1 * rng.standard_normal(
        (X.shape[0], 2)
    )
    # TODO: softmax-normalise logits across axis=1 — shift by max, exp, divide
    exp = ____
    return exp / exp.sum(axis=1, keepdims=True)



## TASK 3 — TRAIN: fit BIC-optimal GMM and extract soft responsibilities


In [ ]:
X_scaled, customers, feature_cols, _ = load_customers_scaled()

best_k, best_bic = None, float("inf")
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42).fit(
        X_scaled
    )
    b = float(gmm.bic(X_scaled))
    if b < best_bic:
        best_bic, best_k = b, k

print("=" * 70)
print(f"  Soft-assignment analysis at BIC-optimal K={best_k}")
print("=" * 70)

# TODO: fit GaussianMixture at best_k with full cov, random_state=42
best_gmm = ____
# TODO: get soft responsibilities with best_gmm.predict_proba(X_scaled)
soft_probs = ____
hard_labels = best_gmm.predict(X_scaled)

profile = soft_vs_hard(soft_probs)
print("\nConfidence bands (fraction of customers in each):")
print(f"  confident  (>0.95):     {profile['confident']:.1%}")
print(f"  moderate   (0.70-0.95): {profile['moderate']:.1%}")
print(f"  ambiguous  (0.50-0.70): {profile['ambiguous']:.1%}")
print(f"  uncertain  (<=0.50):    {profile['uncertain']:.1%}")
print(
    f"\nMean entropy: {profile['mean_entropy']:.4f}  "
    f"(max possible: {profile['max_entropy']:.4f})"
)



### Checkpoint 1


In [ ]:
assert soft_probs.shape == (X_scaled.shape[0], best_k), "R must be (n, K)"
assert abs(soft_probs.sum(axis=1).mean() - 1.0) < 1e-4, "rows must sum to 1"
assert soft_probs.min() >= 0, "responsibilities non-negative"
print("\n[ok] Checkpoint 1 passed — responsibilities form a valid distribution")

# TODO: boundary_mask = soft_probs.max(axis=1) < 0.6
boundary_mask = ____
n_boundary = int(boundary_mask.sum())
print(
    f"\nBoundary customers (max responsibility < 0.6): "
    f"{n_boundary} / {X_scaled.shape[0]} "
    f"({n_boundary / X_scaled.shape[0]:.1%})"
)



## TASK 4 — VISUALISE: confidence bands + segment profile


In [ ]:
viz = ModelVisualizer()
confidence_chart = {
    "Confidence bands": {
        "confident": profile["confident"],
        "moderate": profile["moderate"],
        "ambiguous": profile["ambiguous"],
        "uncertain": profile["uncertain"],
    }
}
fig = viz.metric_comparison(confidence_chart)
fig.update_layout(title="Soft-assignment confidence distribution")
fig.write_html(str(out_path("ex2_soft_confidence.html")))
print(f"\nSaved: {out_path('ex2_soft_confidence.html')}")

# Join labels back onto the customer table for per-segment profiling
customers_with_seg = customers.with_columns(
    pl.Series("gmm_segment", hard_labels),
    pl.Series("gmm_confidence", soft_probs.max(axis=1)),
)

print("\nPer-segment summary (hard counts + soft mass):")
for k in range(best_k):
    subset = customers_with_seg.filter(pl.col("gmm_segment") == k)
    hard_count = subset.height
    soft_mass = float(soft_probs[:, k].sum())
    print(
        f"  Segment {k}: hard={hard_count:>5}  "
        f"soft_mass={soft_mass:>8.1f}  "
        f"weight={best_gmm.weights_[k]:.3f}"
    )



### Checkpoint 2


In [ ]:
assert customers_with_seg.height == X_scaled.shape[0], "join preserved all rows"
assert out_path("ex2_soft_confidence.html").exists(), "confidence chart written"
print("\n[ok] Checkpoint 2 passed — per-segment profile produced")

# MoE gate demo
moe_demo = simple_moe_gate(X_scaled[:, :2])
print("\nMoE gating demo on first 2 features:")
print(f"  Expert 0 active (gate > 0.5): {(moe_demo[:, 0] > 0.5).mean():.1%}")
print(f"  Expert 1 active (gate > 0.5): {(moe_demo[:, 1] > 0.5).mean():.1%}")



### Checkpoint 3


In [ ]:
assert moe_demo.shape == (X_scaled.shape[0], 2), "MoE gate must produce 2 weights"
assert abs(moe_demo.sum(axis=1).mean() - 1.0) < 1e-4, "gate must be a distribution"
print("[ok] Checkpoint 3 passed — MoE gate produces a valid softmax distribution")



## TASK 5 — APPLY: C2C Marketplace Personalised Listing Ranking (Singapore)


In [ ]:
# SCENARIO: A Singapore-based C2C marketplace has (assume) ~35M users.
# The feed ranker has ~80ms to order ~10M live listings per session.
# A Saturday shopper might be 60% bargain hunter, 30% home-decor browser,
# 10% gift shopper in the same session. Hard segment models pick one
# label and miss the other 40% of intent.
#
# Soft responsibilities enable blended ranking:
#   expected_click = sum_k r_k * CTR_k(listing)
# Same idea as Sparse MoE in LLMs: top-K gated experts per query.
#
# BUSINESS IMPACT (illustrative assumptions, not reported figures):
#   - Feed impressions/day: ~900M at baseline CTR ~6.2%
#   - Assume soft intent-vector ranking lifts CTR ~8% vs hard segments
#   - 8% * 900M impressions/day = ~4.5M extra clicks/day
#   - At ~S$0.018 monetisation/click => ~S$81K/day = S$29.6M/year
#   - Zero marginal infra cost: GMM is fitted nightly and responsibilities
#     are materialised into the existing feature store.

print("\n" + "=" * 70)
print("  APPLY — C2C marketplace personalised listing ranking")
print("=" * 70)
print(
    f"Out of {X_scaled.shape[0]} customers, {n_boundary} "
    f"({n_boundary / X_scaled.shape[0]:.1%}) have mixed intent. Soft scoring "
    "keeps ALL of them in the long-tail of every applicable segment ranker."
)
print(
    "At the assumed scale, blending intents with soft responsibilities "
    "is worth an illustrative ~S$29.6M/year in feed monetisation — from the same GMM "
    "you just fitted, read a different way."
)



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# This is the FINAL lesson in the M4 ex_2 GMM block. After it lands the
# m4_clustering_zoo experiment will hold nine runs across five families.

# TODO: call track_run with run_name "gmm_soft_assignment_moe". scalar
# metrics include the four confidence-band percentages from `profile`,
# mean_entropy, max_entropy, boundary_count (n_boundary), boundary_pct,
# and per-expert MoE active rates: float((moe_demo[:, k] > 0.5).mean())
# for k in 0, 1.
track_run(
    tracker,
    exp_name,
    run_name=____,
    params={
        "best_k": best_k,
        "cov_type": "full",
        "n_samples": X_scaled.shape[0],
        "n_features": X_scaled.shape[1],
        "moe_n_experts": 2,
    },
    scalar_metrics={
        "confident_pct": float(profile["confident"]),
        "moderate_pct": float(profile["moderate"]),
        "ambiguous_pct": float(profile["ambiguous"]),
        "uncertain_pct": float(profile["uncertain"]),
        "mean_entropy": float(profile["mean_entropy"]),
        "max_entropy": float(profile["max_entropy"]),
        "boundary_count": float(n_boundary),
        "boundary_pct": float(n_boundary) / float(X_scaled.shape[0]),
        "moe_expert0_active_pct": ____,
        "moe_expert1_active_pct": ____,
    },
)
print(f"  [tracked] soft-assignment + MoE gate metrics logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — engine fit + close the m4_clustering_zoo loop


In [ ]:
# After this lesson, the m4_clustering_zoo experiment in
# mlfp04_ex1_clustering.db holds nine runs across five families:
#   kmeans_pp / hierarchical_<linkage> / dbscan_hdbscan / spectral_rbf
#   evaluation_profiling / em_from_scratch / sklearn_gmm_bic_aic / gmm_cov_<best> / gmm_soft_moe
#
# The leaderboard is the unifying surface across all clustering runs on
# the same Singapore e-commerce dataset. ClusteringEngine.fit returns
# hard labels in one call; soft predict_proba still routes through
# sklearn's GaussianMixture for the responsibility matrix.

from kailash_ml.engines.clustering import ClusteringEngine

cust_df = pl.from_numpy(X_scaled, schema=feature_cols)

# TODO: instantiate ClusteringEngine and call .fit on cust_df with
# algorithm='gmm' and n_clusters=best_k.
clustering = ____
fit_result = ____
print(
    f"  ClusteringEngine.fit(gmm, K={best_k}): "
    f"silhouette={(fit_result.silhouette_score or 0.0):.4f}  "
    f"n_clusters={fit_result.n_clusters}"
)
print(
    "  Open mlfp04_ex1_clustering.db for the full m4_clustering_zoo"
    " leaderboard — nine runs across five clustering families on the"
    " same dataset, ready for cross-method comparison.\n"
)



## REFLECTION


[x] Soft GMM responsibilities carry uncertainty hard labels destroy
  [x] Max-probability and entropy diagnose boundary customers
  [x] MoE = GMM with input-dependent gating g_k(x)
  [x] Sparse MoE in LLMs such as Mixtral is the same idea at LLM scale
  [x] C2C marketplace scenario: soft intent vectors are worth an
      illustrative S$29.6M/year in feed revenue without extra serving
      cost

  KEY INSIGHT: the GMM you just fitted is already a personalisation
  engine. You don't need a new model — you need a new way to READ the
  responsibility matrix. Hard argmax throws away all of the uncertainty.

  Exercise 2 complete. Next: Exercise 3 introduces PCA and
  dimensionality reduction on the same customer data.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)


# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

